CELL 1 - Check GPU
Run this first to confirm you have a GPU available

In [ ]:
import torch

print("Python and PyTorch are ready!")
print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
else:
    print("No GPU found.")
    print("In Colab: Runtime > Change runtime type > T4 GPU")

CELL 2 - Mount Google Drive
Your checkpoints and processed data will be saved here permanently

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/craft-df-training'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/processed_data', exist_ok=True)

print(f"Project directory ready: {PROJECT_DIR}")
print("Your files will be saved here even after the session ends.")

CELL 3 - Clone your repo and install dependencies
Replace YOUR_USERNAME with your actual GitHub username

In [ ]:
import os, sys
from google.colab import userdata

# Reads GITHUB_TOKEN from Colab Secrets
# Secret name must be exactly: GITHUB_TOKEN
token    = userdata.get('GITHUB_TOKEN')
REPO_DIR = "/content/craft-df"
GITHUB_REPO = f"https://{token}@github.com/d1vv1/craft-df.git"

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

!pip install -q -r requirements.txt

import craft_df
print("CRAFT-DF installed successfully!")

CELL 4 - Set up Kaggle using Colab Secrets and download dataset

Reads KAGGLE_USERNAME and KAGGLE_KEY from Colab Secrets (the key icon in the left sidebar).
Make sure both secrets have 'Notebook access' toggled on.

In [ ]:
import os, json
from google.colab import userdata

# Read Kaggle credentials from Colab Secrets
# Secret names must be exactly: KAGGLE_USERNAME and KAGGLE_KEY
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key      = userdata.get('KAGGLE_KEY')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print(f"Kaggle configured for user: {kaggle_username}")

# Download dataset (~2GB)
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip
!ls /content/data/

CELL 5 - Organize data into the required structure

CRAFT-DF expects:
   input_videos/
   ├── real/
   └── fake/

In [ ]:
import os
from pathlib import Path

RAW_DATA_DIR = '/content/data'
INPUT_DIR    = '/content/input_videos'

os.makedirs(f'{INPUT_DIR}/real', exist_ok=True)
os.makedirs(f'{INPUT_DIR}/fake', exist_ok=True)

real_src = f'{RAW_DATA_DIR}/real_and_fake_face/training_real'
fake_src = f'{RAW_DATA_DIR}/real_and_fake_face/training_fake'

if os.path.exists(real_src):
    !cp -r {real_src}/* {INPUT_DIR}/real/
    !cp -r {fake_src}/* {INPUT_DIR}/fake/

real_count = len(list(Path(f'{INPUT_DIR}/real').iterdir()))
fake_count = len(list(Path(f'{INPUT_DIR}/fake').iterdir()))
print(f"Real samples: {real_count}")
print(f"Fake samples: {fake_count}")
print("Data organized successfully!")

CELL 6 - Run data preparation (face detection + DWT processing)
This converts raw videos/images into the .npy format CRAFT-DF needs
Takes 10-60 minutes depending on dataset size

In [ ]:
import os

INPUT_DIR    = '/content/input_videos'
OUTPUT_DIR   = '/content/drive/MyDrive/craft-df-training/processed_data'
METADATA_CSV = '/content/drive/MyDrive/craft-df-training/metadata.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

!python data_prep.py \
    --input_dir {INPUT_DIR} \
    --output_dir {OUTPUT_DIR} \
    --metadata_path {METADATA_CSV} \
    --frame_skip 5 \
    --min_detection_confidence 0.5 \
    --max_frames_per_video 100

import pandas as pd
if os.path.exists(METADATA_CSV):
    df = pd.read_csv(METADATA_CSV)
    print(f"Total samples processed: {len(df)}")
    print(f"Real: {len(df[df['label']==0])}")
    print(f"Fake: {len(df[df['label']==1])}")
    print(df.head())

CELL 7 - Create training configuration
Tuned for Colab's T4 GPU (15GB VRAM)

In [ ]:
import yaml, os

config = {
    'model': {
        'spatial_backbone': 'mobilenet_v2',
        'spatial_pretrained': True,
        'spatial_freeze_layers': 10,
        'freq_dwt_levels': 3,
        'freq_wavelet': 'db4',
        'attention_heads': 8,
        'attention_dim': 512,
        'dropout_rate': 0.1,
    },
    'training': {
        'learning_rate': 1e-4,
        'batch_size': 32,
        'max_epochs': 30,
        'num_workers': 2,
        'pin_memory': True,
        'gradient_clip_val': 1.0,
        'accumulate_grad_batches': 1,
        'precision': 16,
    },
    'data': {
        'input_size': [224, 224],
        'dwt_levels': 3,
        'wavelet_type': 'db4',
        'face_confidence_threshold': 0.5,
        'train_split': 0.7,
        'val_split': 0.15,
        'test_split': 0.15,
        'metadata_path': '/content/drive/MyDrive/craft-df-training/metadata.csv',
    },
    'logging': {
        'project_name': 'craft-df',
        'experiment_name': 'colab-run-1',
        'log_every_n_steps': 10,
        'save_top_k': 3,
        'monitor': 'val_accuracy',
        'mode': 'max',
    },
    'reproducibility': {
        'seed': 42,
        'deterministic': True,
        'benchmark': False,
    },
    'hardware': {
        'accelerator': 'gpu',
        'devices': 1,
        'strategy': 'auto',
        'sync_batchnorm': False,
    },
}

os.makedirs('configs', exist_ok=True)
with open('configs/colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config saved to configs/colab.yaml")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Max epochs: {config['training']['max_epochs']}")
print(f"Mixed precision: {config['training']['precision'] == 16}")

CELL 8 - Set up Weights & Biases using Colab Secrets

Reads WANDB_API_KEY from Colab Secrets (the key icon in the left sidebar).
Make sure the secret has 'Notebook access' toggled on.

In [ ]:
import wandb
from google.colab import userdata

# Read W&B API key from Colab Secrets
# Secret name must be exactly: WANDB_API_KEY
wandb_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_key)

print("W&B ready. Your training charts will appear at wandb.ai")

CELL 9 - START TRAINING
Watch the loss go down!
Expected time on T4 GPU: ~20-40 min for 30 epochs on a small dataset

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/craft-df-training/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!python train.py \
    --config configs/colab.yaml \
    --experiment colab-run-1

print("Training complete!")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")

CELL 10 - Check training results

In [ ]:
import os
from pathlib import Path

CHECKPOINT_DIR = '/content/drive/MyDrive/craft-df-training/checkpoints'

checkpoints = list(Path(CHECKPOINT_DIR).glob('*.ckpt'))
if checkpoints:
    print(f"Saved {len(checkpoints)} checkpoints:")
    for ckpt in sorted(checkpoints):
        size_mb = ckpt.stat().st_size / 1e6
        print(f"  {ckpt.name}  ({size_mb:.1f} MB)")
else:
    print("No checkpoints found yet.")

best_ckpt = CHECKPOINT_DIR + '/best.ckpt'
if os.path.exists(best_ckpt):
    print(f"\nBest model: {best_ckpt}")
    print("You can now use this for inference!")

CELL 11 - Quick inference test on a sample video

In [ ]:
import torch, sys
sys.path.insert(0, '/content/craft-df')

from craft_df.models.craft_df_model import CRAFTDFModel

CHECKPOINT_PATH = '/content/drive/MyDrive/craft-df-training/checkpoints/best.ckpt'

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
model = CRAFTDFModel()
model.load_state_dict(checkpoint['state_dict'])
model.eval()

print("Model loaded successfully!")
print("Ready for inference on new videos.")

# To run inference on a video:
# from examples.model_inference import CRAFTDFInference
# inference = CRAFTDFInference(CHECKPOINT_PATH)
# result = inference.predict_video('/path/to/video.mp4')
# print(result)

CELL 12 - Resume training if session disconnected
Colab disconnects after ~12 hours. Re-run cells 2, 3, 7, 8 first, then run this.

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/craft-df-training/checkpoints'
last_ckpt = CHECKPOINT_DIR + '/last.ckpt'

if os.path.exists(last_ckpt):
    print(f"Resuming from: {last_ckpt}")
    !python train.py \
        --config configs/colab.yaml \
        --experiment colab-run-1-resumed \
        --resume {last_ckpt}
else:
    print("No checkpoint found to resume from. Run Cell 9 first.")